# Dimensionless_Hyperliquid_backtester

# Hyperliquid RSI Backtest Research Framework 

This notebook is designed for **historical research only**; it does not place live orders.

## Important modelling choices
- Long-only RSI threshold-crossing strategy (the original strategy idea is preserved).
- Signals use a completed candle and are executed no earlier than the **next candle open**.
- Linear-perpetual-style P&L: signed quantity × price change. Collateral/equity is tracked separately from notional exposure.
- Entry and exit fees and configurable adverse slippage are charged on executed notional.
- Stop/target checks use candle high/low. If both stop and target are touched in one candle, the conservative stop is assumed first.
- Funding is **not** inferred. Supply a funding-rate series if available or interpret results as excluding funding.
- Liquidation is a configurable simplified maintenance-margin approximation, not a reproduction of Hyperliquid's full account-level liquidation engine.
- Historical data pagination and coverage are checked, but exchange availability/rate limits can still restrict the requested history.

Do not interpret results as evidence of profitability until the data, costs, assumptions, and out-of-sample tests have been reviewed.

## 1. Imports and configuration

In [2]:
from __future__ import annotations
from dataclasses import dataclass, asdict
from typing import Optional, Iterable
import math
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

@dataclass(frozen=True)
class BacktestConfig:
    symbol: str = "BTC/USDC:USDC"
    timeframe: str = "1d"
    data_start: str = "2021-01-01"
    data_end: str = "2026-01-01"       # exclusive upper bound
    backtest_start: str = "2022-12-01"
    backtest_end: str = "2026-01-01"   # exclusive upper bound
    initial_capital: float = 1000.0
    allocation_pct: float = 50.0       # fraction of equity allocated as initial margin
    leverage: float = 1.0
    fee_rate: float = 0.00045          # 0.045% per side, decimal not percent
    slippage_bps: float = 2.0          # adverse slippage per fill, basis points
    rsi_length: int = 14
    rsi_threshold: float = 70.0
    stop_loss_pct: Optional[float] = None  # e.g. 0.10 for 10%; disabled by None
    take_profit_pct: Optional[float] = None
    maintenance_margin_rate: float = 0.005

    def validate(self):
        if self.initial_capital <= 0: raise ValueError("initial_capital must be > 0")
        if not 0 < self.allocation_pct <= 100: raise ValueError("allocation_pct must be in (0, 100]")
        if self.leverage < 1: raise ValueError("leverage must be >= 1")
        if self.fee_rate < 0 or self.slippage_bps < 0: raise ValueError("fees/slippage cannot be negative")
        if self.rsi_length < 2: raise ValueError("rsi_length must be >= 2")
        if not 0 < self.rsi_threshold < 100: raise ValueError("rsi_threshold must be between 0 and 100")
        for name in ("stop_loss_pct", "take_profit_pct"):
            value = getattr(self, name)
            if value is not None and not 0 < value < 1: raise ValueError(f"{name} must be a fraction in (0, 1), e.g. 0.05")
        if not 0 <= self.maintenance_margin_rate < 1: raise ValueError("maintenance_margin_rate must be in [0, 1)")
        if pd.Timestamp(self.data_start) >= pd.Timestamp(self.data_end): raise ValueError("data_start must precede data_end")
        if pd.Timestamp(self.backtest_start) >= pd.Timestamp(self.backtest_end): raise ValueError("backtest_start must precede backtest_end")

CONFIG = BacktestConfig()
CONFIG.validate()
pd.set_option("display.max_columns", 100)
print(asdict(CONFIG))

{'symbol': 'BTC/USDC:USDC', 'timeframe': '1d', 'data_start': '2021-01-01', 'data_end': '2026-01-01', 'backtest_start': '2022-12-01', 'backtest_end': '2026-01-01', 'initial_capital': 1000.0, 'allocation_pct': 50.0, 'leverage': 1.0, 'fee_rate': 0.00045, 'slippage_bps': 2.0, 'rsi_length': 14, 'rsi_threshold': 70.0, 'stop_loss_pct': None, 'take_profit_pct': None, 'maintenance_margin_rate': 0.005}


## 2. Historical market data: pagination, validation, coverage

In [3]:
OHLCV_COLUMNS = ["timestamp", "open", "high", "low", "close", "volume"]

def validate_ohlcv(data: pd.DataFrame, *, require_nonempty: bool = True) -> pd.DataFrame:
    """Return a clean, chronologically sorted OHLCV frame; reject malformed prices."""
    if data is None or data.empty:
        if require_nonempty: raise ValueError("OHLCV data is empty")
        return pd.DataFrame(columns=OHLCV_COLUMNS)
    missing = set(OHLCV_COLUMNS) - set(data.columns)
    if missing: raise ValueError(f"Missing OHLCV columns: {sorted(missing)}")
    out = data[OHLCV_COLUMNS].copy()
    out["timestamp"] = pd.to_datetime(out["timestamp"], utc=True).dt.tz_convert(None)
    for col in OHLCV_COLUMNS[1:]: out[col] = pd.to_numeric(out[col], errors="coerce")
    if out[OHLCV_COLUMNS[1:]].isna().any().any(): raise ValueError("OHLCV contains NaN/non-numeric values")
    if not np.isfinite(out[OHLCV_COLUMNS[1:]].to_numpy(dtype=float)).all(): raise ValueError("OHLCV contains non-finite values")
    if (out[["open", "high", "low", "close"]] <= 0).any().any(): raise ValueError("OHLC prices must be positive")
    if (out["volume"] < 0).any(): raise ValueError("Volume cannot be negative")
    if (out["high"] < out[["open", "low", "close"]].max(axis=1)).any(): raise ValueError("Invalid OHLC: high below open/low/close")
    if (out["low"] > out[["open", "high", "close"]].min(axis=1)).any(): raise ValueError("Invalid OHLC: low above open/high/close")
    out = out.drop_duplicates("timestamp", keep="last").sort_values("timestamp").reset_index(drop=True)
    if out["timestamp"].duplicated().any(): raise AssertionError("Timestamp deduplication failed")
    return out

def download_data(symbol: str, timeframe: str, start: str, end: str, *, limit: int = 500) -> pd.DataFrame:
    """Paginate CCXT OHLCV requests over [start, end); does not guarantee exchange completeness."""
    try:
        import ccxt
    except ImportError as exc:
        raise ImportError("Install ccxt to download data: pip install ccxt") from exc
    start_ts = pd.Timestamp(start, tz="UTC")
    end_ts = pd.Timestamp(end, tz="UTC")
    if start_ts >= end_ts: raise ValueError("start must precede end")
    exchange = ccxt.hyperliquid({"enableRateLimit": True})
    since_ms = int(start_ts.timestamp() * 1000)
    end_ms = int(end_ts.timestamp() * 1000)
    rows, seen = [], set()
    # A hard cap protects against a broken exchange response that never advances.
    for _ in range(10000):
        batch = exchange.fetch_ohlcv(symbol, timeframe=timeframe, since=since_ms, limit=limit)
        if not batch: break
        fresh = []
        for candle in batch:
            ts = int(candle[0])
            if ts >= end_ms: continue
            if ts >= int(start_ts.timestamp() * 1000) and ts not in seen:
                rows.append(candle); seen.add(ts); fresh.append(ts)
        max_ts = max(int(c[0]) for c in batch)
        if max_ts >= end_ms or max_ts < since_ms: break
        next_since = max_ts + 1
        if next_since <= since_ms: raise RuntimeError("OHLCV pagination made no progress")
        since_ms = next_since
    else:
        raise RuntimeError("OHLCV pagination exceeded safety cap")
    if not rows: raise ValueError(f"No OHLCV candles returned for {symbol} in requested range")
    frame = pd.DataFrame(rows, columns=OHLCV_COLUMNS)
    frame["timestamp"] = pd.to_datetime(frame["timestamp"], unit="ms", utc=True).dt.tz_convert(None)
    frame = validate_ohlcv(frame)
    actual_start, actual_end = frame.timestamp.min(), frame.timestamp.max()
    print(f"Downloaded {len(frame):,} candles; actual coverage {actual_start} through {actual_end} (end exclusive request: {end}).")
    if actual_start > start_ts.tz_convert(None): warnings.warn(f"Data starts later than requested: {actual_start} > {start}")
    if actual_end < (end_ts - pd.Timedelta(days=1)).tz_convert(None) and timeframe == "1d":
        warnings.warn(f"Daily data ends earlier than expected: {actual_end}; check listing/API coverage and end date")
    return frame

# Run this cell to fetch fresh data. If you already have a validated local dataset, assign it to raw_data instead.
# raw_data = download_data(CONFIG.symbol, CONFIG.timeframe, CONFIG.data_start, CONFIG.data_end)

## 3. Indicators and strategy signals

In [4]:
def rsi_wilder(close: pd.Series, length: int = 14) -> pd.Series:
    """Wilder RSI; output is aligned to close and is NaN during warm-up."""
    close = pd.to_numeric(close, errors="raise").astype(float)
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1 / length, min_periods=length, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / length, min_periods=length, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    result = 100 - (100 / (1 + rs))
    result = result.mask((avg_loss == 0) & (avg_gain > 0), 100.0)
    result = result.mask((avg_gain == 0) & (avg_loss == 0), 50.0)
    return result.rename("rsi")

def prepare_data(raw_data: pd.DataFrame, config: BacktestConfig = CONFIG) -> pd.DataFrame:
    """Validate full warm-up history, calculate RSI, then select [backtest_start, backtest_end)."""
    config.validate()
    frame = validate_ohlcv(raw_data)
    frame["rsi"] = rsi_wilder(frame["close"], config.rsi_length)
    start, end = pd.Timestamp(config.backtest_start), pd.Timestamp(config.backtest_end)
    selected = frame.loc[(frame.timestamp >= start) & (frame.timestamp < end)].copy()
    selected = selected.dropna(subset=["rsi"]).reset_index(drop=True)
    if len(selected) < 3: raise ValueError("Not enough valid candles in the backtest window after indicator warm-up")
    if selected.timestamp.duplicated().any() or not selected.timestamp.is_monotonic_increasing:
        raise ValueError("Prepared data must have unique ascending timestamps")
    return selected

def generate_signals(data: pd.DataFrame, config: BacktestConfig = CONFIG) -> pd.DataFrame:
    """Signals are known at each candle close; the simulator executes them on the next bar."""
    out = data.copy()
    prev = out["rsi"].shift(1)
    out["entry_signal"] = (prev <= config.rsi_threshold) & (out["rsi"] > config.rsi_threshold)
    out["exit_signal"] = (prev >= config.rsi_threshold) & (out["rsi"] < config.rsi_threshold)
    out["entry_signal"] = out["entry_signal"].fillna(False).astype(bool)
    out["exit_signal"] = out["exit_signal"].fillna(False).astype(bool)
    return out

## 4. Event-driven backtest engine

In [5]:
def run_backtest(data: pd.DataFrame, config: BacktestConfig = CONFIG, funding_rates: Optional[pd.Series] = None):
    """Long-only (unless enabled) linear-perpetual simulation.

    funding_rates, if provided, must be a Series indexed by candle timestamp with a decimal
    funding rate per candle (positive means longs pay shorts). Funding is applied to the
    position notional at each candle close. If absent, funding is excluded and reported as zero.
    """
    config.validate()
    if "rsi" not in data.columns:
        raise ValueError("Input data must include an rsi column; call prepare_data first")
    frame = data.copy()
    validated = validate_ohlcv(frame)
    # Preserve indicator columns while taking canonical, validated OHLCV columns.
    extras = frame.drop(columns=[c for c in OHLCV_COLUMNS if c != "timestamp" and c in frame.columns]).copy()
    frame = validated.merge(extras, on="timestamp", how="left", validate="one_to_one")
    frame = generate_signals(frame, config).reset_index(drop=True)
    if frame["rsi"].isna().any(): raise ValueError("Input data must include fully populated rsi values")
    if len(frame) < 2: raise ValueError("At least two candles are required")
    if funding_rates is not None:
        funding_rates = pd.Series(funding_rates, dtype=float).copy()
        funding_rates.index = pd.to_datetime(funding_rates.index)
        if not np.isfinite(funding_rates.dropna().to_numpy()).all(): raise ValueError("Funding rates must be finite")

    cash = float(config.initial_capital)  # collateral/equity excluding open-position unrealized P&L
    position = None
    events, trades, curve = [], [], []
    total_funding = 0.0
    trade_id = 0

    def equity_at(mark):
        return cash + (position["qty"] * (mark - position["entry_price"]) if position else 0.0)

    def record_event(ts, event, side, price, qty, fee=0.0, pnl=0.0, reason=None):
        events.append({"timestamp": ts, "event": event, "side": side, "price": float(price),
                       "quantity": float(qty), "notional": float(abs(price * qty)), "fee": float(fee),
                       "pnl": float(pnl), "cash_after": float(cash), "equity_after": float(equity_at(price)),
                       "reason": reason})

    def close_position(ts, raw_price, reason, adverse_direction):
        nonlocal cash, position
        pos = position
        # adverse_direction: +1 for selling long (slippage lowers proceeds), -1 for buying short (not used by default)
        fill = raw_price * (1 - config.slippage_bps / 10000) if pos["side"] == "long" else raw_price * (1 + config.slippage_bps / 10000)
        qty_signed = pos["qty"]
        gross = qty_signed * (fill - pos["entry_price"])
        fee = abs(qty_signed * fill) * config.fee_rate
        cash += gross - fee
        net_trade_pnl = gross - pos["entry_fee"] - fee - pos["funding_paid"]
        trades.append({"trade_id": pos["trade_id"], "side": pos["side"], "entry_time": pos["entry_time"],
                       "exit_time": ts, "entry_price": pos["entry_price"], "exit_price": fill,
                       "quantity": qty_signed, "entry_notional": abs(qty_signed * pos["entry_price"]),
                       "gross_pnl": gross, "entry_fee": pos["entry_fee"], "exit_fee": fee,
                       "funding_paid": pos["funding_paid"], "net_pnl": net_trade_pnl,
                       "return_on_entry_notional_pct": 100 * net_trade_pnl / max(abs(qty_signed * pos["entry_price"]), 1e-12),
                       "holding_bars": pos["bars_held"], "exit_reason": reason})
        position = None
        record_event(ts, "exit", pos["side"], fill, qty_signed, fee, gross, reason)

    for i, row in frame.iterrows():
        ts = row["timestamp"]
        # Decisions from bar i-1 are available at bar i open. Never use bar i close to fill an order.
        if i > 0 and position is not None and bool(frame.iloc[i-1]["exit_signal"]):
            close_position(ts, float(row["open"]), "rsi_exit_next_open", 1)

        # Open a new position at this bar's open only when the previous completed bar signalled entry.
        if i > 0 and position is None and bool(frame.iloc[i-1]["entry_signal"]):
            entry_raw = float(row["open"])
            entry_fill = entry_raw * (1 + config.slippage_bps / 10000)
            equity_before = equity_at(entry_fill)
            margin_budget = max(0.0, equity_before) * config.allocation_pct / 100
            notional = margin_budget * config.leverage
            qty = notional / entry_fill
            if qty > 0 and notional > 0:
                trade_id += 1
                fee_entry = notional * config.fee_rate
                cash -= fee_entry
                position = {"trade_id": trade_id, "side": "long", "qty": qty, "entry_price": entry_fill,
                            "entry_time": ts, "entry_fee": fee_entry, "funding_paid": 0.0, "bars_held": 0}
                record_event(ts, "entry", "long", entry_fill, qty, fee_entry, 0.0, "rsi_cross_up")

        # Intrabar risk controls. If both SL and TP are crossed, choose the stop conservatively.
        if position is not None:
            position["bars_held"] += 1
            entry = position["entry_price"]
            sl = entry * (1 - config.stop_loss_pct) if config.stop_loss_pct is not None else None
            tp = entry * (1 + config.take_profit_pct) if config.take_profit_pct is not None else None
            hit_sl = sl is not None and float(row["low"]) <= sl
            hit_tp = tp is not None and float(row["high"]) >= tp
            if hit_sl:
                close_position(ts, float(sl), "stop_loss" if not hit_tp else "stop_loss_ambiguous_bar", 1)
            elif hit_tp:
                close_position(ts, float(tp), "take_profit", 1)

        # Optional funding: rate is decimal per candle; positive rates are paid by long positions.
        if position is not None and funding_rates is not None:
            rate = funding_rates.get(pd.Timestamp(ts), 0.0)
            payment = abs(position["qty"] * float(row["close"])) * float(rate)
            cash -= payment
            position["funding_paid"] += payment
            total_funding += payment

        mark = float(row["close"])
        unrealized = position["qty"] * (mark - position["entry_price"]) if position else 0.0
        equity = cash + unrealized
        notional_mark = abs(position["qty"] * mark) if position else 0.0
        margin_ratio = (equity / notional_mark) if notional_mark > 0 else np.nan
        # Approximate maintenance-margin breach; not an exchange-exact liquidation simulation.
        if position is not None and equity <= notional_mark * config.maintenance_margin_rate:
            close_position(ts, mark, "maintenance_margin_breach_approx", 1)
            equity = cash
            unrealized = 0.0
            notional_mark = 0.0
            margin_ratio = np.nan
        curve.append({"timestamp": ts, "cash": cash, "unrealized_pnl": unrealized, "equity": equity,
                      "position_qty": position["qty"] if position else 0.0,
                      "position_notional": notional_mark, "margin_ratio": margin_ratio,
                      "funding_cumulative": total_funding, "close": mark})

    # Mark-to-market open position remains open; no artificial forced close at end of sample.
    curve = pd.DataFrame(curve)
    curve["equity_peak"] = curve["equity"].cummax()
    curve["drawdown"] = curve["equity"] / curve["equity_peak"] - 1.0
    curve["returns"] = curve["equity"].pct_change().replace([np.inf, -np.inf], np.nan).fillna(0.0)
    curve["buy_hold_equity"] = config.initial_capital * curve["close"] / curve["close"].iloc[0]
    events_df = pd.DataFrame(events, columns=["timestamp", "event", "side", "price", "quantity", "notional", "fee", "pnl", "cash_after", "equity_after", "reason"])
    trades_df = pd.DataFrame(trades, columns=["trade_id", "side", "entry_time", "exit_time", "entry_price", "exit_price", "quantity", "entry_notional", "gross_pnl", "entry_fee", "exit_fee", "funding_paid", "net_pnl", "return_on_entry_notional_pct", "holding_bars", "exit_reason"])
    return curve, events_df, trades_df

## 5. Performance analysis

In [6]:
def performance_report(curve: pd.DataFrame, trades: pd.DataFrame, config: BacktestConfig = CONFIG, events: Optional[pd.DataFrame] = None) -> pd.Series:
    """Metrics from the equity curve and completed round-trip trades; annualization uses observed timestamps."""
    if curve.empty: raise ValueError("Equity curve is empty")
    equity = curve["equity"].astype(float)
    elapsed_days = max((pd.Timestamp(curve.timestamp.iloc[-1]) - pd.Timestamp(curve.timestamp.iloc[0])).total_seconds() / 86400, 0)
    total_return = equity.iloc[-1] / config.initial_capital - 1
    years = elapsed_days / 365.25
    cagr = (equity.iloc[-1] / config.initial_capital) ** (1 / years) - 1 if years > 0 and equity.iloc[-1] > 0 else np.nan
    rets = curve["returns"].astype(float).dropna()
    # Estimate periods/year from median observed interval; this also works beyond daily bars.
    dt = pd.to_datetime(curve.timestamp).diff().dt.total_seconds().dropna()
    periods_per_year = 365.25 * 86400 / dt.median() if len(dt) and dt.median() > 0 else np.nan
    vol = rets.std(ddof=1) * math.sqrt(periods_per_year) if len(rets) > 1 and np.isfinite(periods_per_year) else np.nan
    sharpe = (rets.mean() / rets.std(ddof=1) * math.sqrt(periods_per_year)) if len(rets) > 1 and rets.std(ddof=1) > 0 and np.isfinite(periods_per_year) else np.nan
    downside = rets[rets < 0]
    sortino = (rets.mean() / downside.std(ddof=1) * math.sqrt(periods_per_year)) if len(downside) > 1 and downside.std(ddof=1) > 0 and np.isfinite(periods_per_year) else np.nan
    max_dd = float(curve["drawdown"].min())
    wins = trades.loc[trades.net_pnl > 0, "net_pnl"] if not trades.empty else pd.Series(dtype=float)
    losses = trades.loc[trades.net_pnl < 0, "net_pnl"] if not trades.empty else pd.Series(dtype=float)
    gross_profit, gross_loss = wins.sum(), abs(losses.sum())
    pf = gross_profit / gross_loss if gross_loss > 0 else (np.inf if gross_profit > 0 else np.nan)
    ntrades = len(trades)
    fees = float(events.loc[events.event.isin(["entry", "exit"]), "fee"].sum()) if events is not None and not events.empty else (float(trades.entry_fee.sum() + trades.exit_fee.sum()) if not trades.empty else 0.0)
    return pd.Series({"initial_capital": config.initial_capital, "final_equity": equity.iloc[-1],
        "net_profit": equity.iloc[-1] - config.initial_capital, "total_return_pct": total_return * 100,
        "CAGR_pct": cagr * 100 if np.isfinite(cagr) else np.nan, "annualized_volatility_pct": vol * 100 if np.isfinite(vol) else np.nan,
        "Sharpe_0rf": sharpe, "Sortino_0rf": sortino, "max_drawdown_pct": max_dd * 100,
        "completed_trades": ntrades, "win_rate_pct": (trades.net_pnl.gt(0).mean() * 100) if ntrades else np.nan,
        "average_trade_net_pnl": trades.net_pnl.mean() if ntrades else np.nan,
        "profit_factor": pf, "total_entry_exit_fees": fees,
        "buy_hold_final_equity": curve.buy_hold_equity.iloc[-1],
        "buy_hold_return_pct": (curve.buy_hold_equity.iloc[-1] / config.initial_capital - 1) * 100,
        "period_days": elapsed_days})

def plot_results(curve: pd.DataFrame):
    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    axes[0].plot(curve.timestamp, curve.equity, label="Strategy equity")
    axes[0].plot(curve.timestamp, curve.buy_hold_equity, label="Buy & hold (gross price return)")
    axes[0].set_ylabel("Equity (quote currency)"); axes[0].legend(); axes[0].grid(True, alpha=.25)
    axes[1].fill_between(curve.timestamp, curve.drawdown * 100, 0)
    axes[1].set_ylabel("Drawdown (%)"); axes[1].set_xlabel("Time"); axes[1].grid(True, alpha=.25)
    plt.tight_layout(); return fig

def cost_sensitivity(data: pd.DataFrame, config: BacktestConfig = CONFIG, scenarios=None) -> pd.DataFrame:
    """Re-run a fixed strategy under alternative fee/slippage assumptions; no parameter tuning."""
    from dataclasses import replace
    if scenarios is None:
        scenarios = [(0.00045, 2.0), (0.00075, 5.0), (0.00100, 10.0)]
    rows = []
    for fee_rate, slippage_bps in scenarios:
        cfg = replace(config, fee_rate=float(fee_rate), slippage_bps=float(slippage_bps))
        cfg.validate()
        curve_i, events_i, trades_i = run_backtest(data, cfg)
        report_i = performance_report(curve_i, trades_i, cfg, events_i)
        rows.append({"fee_rate_pct_per_side": fee_rate * 100, "slippage_bps_per_side": slippage_bps,
                     "final_equity": report_i["final_equity"], "total_return_pct": report_i["total_return_pct"],
                     "max_drawdown_pct": report_i["max_drawdown_pct"], "completed_trades": report_i["completed_trades"],
                     "total_fees": report_i["total_entry_exit_fees"]})
    return pd.DataFrame(rows)

def calendar_year_returns(curve: pd.DataFrame) -> pd.DataFrame:
    """Calendar-year returns; first/last partial years are partial periods."""
    if curve.empty: return pd.DataFrame(columns=["year", "start_equity", "end_equity", "return_pct", "partial_period"])
    frame = curve[["timestamp", "equity"]].copy()
    frame["year"] = pd.to_datetime(frame.timestamp).dt.year
    first_year, last_year = int(frame.year.iloc[0]), int(frame.year.iloc[-1])
    rows = []
    for year, group in frame.groupby("year", sort=True):
        start_eq, end_eq = float(group.equity.iloc[0]), float(group.equity.iloc[-1])
        rows.append({"year": int(year), "start_equity": start_eq, "end_equity": end_eq,
                     "return_pct": (end_eq / start_eq - 1) * 100 if start_eq else np.nan,
                     "partial_period": int(year) == first_year or int(year) == last_year})
    return pd.DataFrame(rows)


## 6. Run the backtest

Uncomment the download line to fetch data, or load a locally stored OHLCV CSV with columns `timestamp, open, high, low, close, volume`. Do not use a CSV unless its provenance, timezone, and completeness are known.

In [7]:
# Option A: download from Hyperliquid (requires network access and ccxt)
# raw_data = download_data(CONFIG.symbol, CONFIG.timeframe, CONFIG.data_start, CONFIG.data_end)

# Option B: load your own validated data
# raw_data = pd.read_csv("ohlcv.csv", parse_dates=["timestamp"])

# Once raw_data is available, run:
# prepared = prepare_data(raw_data, CONFIG)
# curve, events, trades = run_backtest(prepared, CONFIG)
# report = performance_report(curve, trades, CONFIG)
# display(report.to_frame("value"))
# display(trades.head(20))
# display(events.tail(20))
# plot_results(curve)

## 7. Validation tests (synthetic, deterministic)

In [8]:
def run_validation_tests():
    # Validate OHLC data checks and RSI bounds / warm-up.
    ts = pd.date_range("2024-01-01", periods=80, freq="D")
    close = pd.Series(100 + np.arange(80) * 0.5 + np.sin(np.arange(80) / 3) * 3)
    synthetic = pd.DataFrame({"timestamp": ts, "open": close, "high": close + 2, "low": close - 2,
                              "close": close, "volume": 1000.0})
    clean = validate_ohlcv(synthetic)
    assert len(clean) == 80 and clean.timestamp.is_monotonic_increasing
    rsi = rsi_wilder(clean.close, 14)
    assert rsi.dropna().between(0, 100).all()
    assert rsi.iloc[:14].isna().all()

    # Explicit portfolio/accounting sanity check: one long with entry/exit fees.
    cfg = BacktestConfig(initial_capital=1000, allocation_pct=50, leverage=1, fee_rate=0.001,
                         slippage_bps=0, rsi_length=2, rsi_threshold=50,
                         stop_loss_pct=None, take_profit_pct=None, maintenance_margin_rate=0.005)
    # Force one entry signal on bar 0; exit signal on bar 2. Simulator executes at next opens.
    test = pd.DataFrame({"timestamp": pd.date_range("2024-01-01", periods=4, freq="D"),
                         "open": [100., 100., 110., 120.], "high": [101., 111., 121., 121.],
                         "low": [99., 99., 109., 119.], "close": [100., 110., 120., 120.],
                         "volume": [1e6]*4, "rsi": [71., 72., 60., 55.]})
    # This strategy signal generator derives signals from RSI, so ensure crossing pattern.
    test["rsi"] = [40., 75., 40., 35.]
    c2, e2, t2 = run_backtest(test, cfg)
    assert len(e2) >= 2, "Expected entry and exit events from synthetic signal sequence"
    assert (e2.fee >= 0).all()
    assert np.isfinite(c2.equity).all()
    assert np.isfinite(c2.drawdown).all()
    assert (c2.drawdown <= 1e-12).all(), "Drawdown should never be positive"
    assert c2.equity.iloc[-1] > 0
    # Stops and targets are valid fractions; invalid configuration must fail.
    try:
        BacktestConfig(stop_loss_pct=1.2).validate()
        raise AssertionError("Invalid stop-loss fraction was not rejected")
    except ValueError:
        pass
    return {"status": "passed", "ohlcv_rows": len(clean), "rsi_valid_values": int(rsi.notna().sum()),
            "synthetic_events": len(e2), "synthetic_completed_trades": len(t2)}

run_validation_tests()

{'status': 'passed',
 'ohlcv_rows': 80,
 'rsi_valid_values': 66,
 'synthetic_events': 2,
 'synthetic_completed_trades': 1}

## 8. Research checklist before interpreting results

- [ ] Verify requested and actual candle coverage; investigate missing daily bars rather than blindly filling them.
- [ ] Confirm the exchange symbol maps to the intended linear perpetual contract and quote currency.
- [ ] Confirm the applicable maker/taker fee tier and slippage assumptions.
- [ ] Add funding history for perpetuals; current results exclude funding when no series is supplied.
- [ ] Validate the maintenance-margin/liquidation approximation against exchange rules before leverage studies.
- [ ] Compare against buy-and-hold using matching dates and clearly labelled gross/net assumptions.
- [ ] Freeze the strategy before evaluating an untouched out-of-sample period.
- [ ] Use walk-forward testing, parameter stability, and block-bootstrap confidence intervals where appropriate.
- [ ] Stress higher fees/slippage, gaps, missing data, and adverse market regimes.
- [ ] Run paper trading with exchange-state reconciliation before any live deployment.

### Known limitations
This is a much safer research baseline, not an exchange-exact simulator. It does not model order-book depth, partial fills, latency, funding unless supplied, cross-margin interactions, tiered maintenance margin, insurance/ADL, or the full exchange liquidation process. Buy-and-hold is shown as gross price return. The default data-download cell is intentionally commented out so opening the notebook does not make network requests automatically.